# AI Job Market 2024–2030: Data Exploration & Quality Findings

This notebook documents the initial data exploration and quality findings for the **AI Impact on Job Market: Increasing vs Decreasing Jobs (2024–2030)** dataset.

The primary analysis and interactive dashboards for this dataset were built in Tableau Public. This notebook serves as a companion, covering:
- Standard data exploration (shape, types, missing values, distributions)
- Establishing the data grain
- Documenting a data quality finding in the `Job Status` field
- Deriving and validating the **Total Structural Change** metric

**Interactive Tableau Story:** [AI & The Future of Work: 2024–2030](https://public.tableau.com/app/profile/robert.wilson8303/viz/ai_jobmarket/AITheFutureofWorkADataStory)

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset — works in both Kaggle and local/GitHub environments
try:
    # Kaggle environment
    df = pd.read_csv('/kaggle/input/datasets/sahilislam007/ai-impact-on-job-market-20242030/ai_job_trends_dataset.csv')
except FileNotFoundError:
    # Local / GitHub environment
    df = pd.read_csv('../data/ai_job_trends_dataset.csv')

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')

## 2. Basic Data Exploration

In [ ]:
# First look at the data
df.head()

In [ ]:
# Data types per column
df.dtypes

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

In [ ]:
# Basic statistics for numeric columns
df.describe()

In [ ]:
# Unique value counts per column — useful for spotting categorical fields
print('Unique values per column:')
print(df.nunique())

In [ ]:
# Distribution of categorical fields
categorical_cols = ['Industry', 'Job Status', 'AI Impact Level', 'Required Education', 'Location']

for col in categorical_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 3. Establishing the Data Grain

Before any analysis, it is important to understand the **grain** of the dataset — the level at which each row is unique. This determines how aggregations behave and which combinations of columns are needed to identify a specific data point.

We test progressively, adding one dimension at a time until no duplicates remain.

In [ ]:
# Test grain progressively
grain_candidates = [
    ['Job Title'],
    ['Job Title', 'Industry'],
    ['Job Title', 'Industry', 'Location'],
    ['Job Title', 'Industry', 'Location', 'Experience Required (Years)']
]

for cols in grain_candidates:
    dupes = df.duplicated(subset=cols).sum()
    print(f'Columns: {cols}')
    print(f'  Duplicates: {dupes} ({dupes/len(df)*100:.1f}% of rows)\n')

In [ ]:
# Confirm the grain
grain = ['Job Title', 'Industry', 'Location', 'Experience Required (Years)']
total_dupes = df.duplicated(subset=grain).sum()

if total_dupes == 0:
    print(f'Grain confirmed: {grain}')
    print(f'Each combination of Job Title + Industry + Location + Experience is unique.')
else:
    print(f'WARNING: {total_dupes} duplicates still exist. Grain may require additional columns.')

513 duplicates remain even after including all four candidate dimensions. Let's investigate what differs between these duplicate rows.

In [ ]:
# Find duplicate rows and examine what differs between them
dupes_mask = df.duplicated(subset=grain, keep=False)
dupe_rows = df[dupes_mask].sort_values(grain)

print(f'Total rows involved in duplicates: {len(dupe_rows)}')
dupe_rows.head(20)

In [ ]:
# How many unique grain combinations have duplicates, and what is the max repetition?
dupe_count = df.groupby(grain).size()
print(f'Grain combinations appearing more than once: {(dupe_count > 1).sum()}')
print(f'Max occurrences of same grain combination: {dupe_count.max()}')
print(f'\nConclusion: The dataset contains {total_dupes} rows where identical grain')
print(f'combinations have different measure values across almost all columns')
print(f'(Job Status, AI Impact Level, Median Salary, Job Openings, Automation Risk etc.).')
print(f'This is almost certainly a synthetic data generation artefact — the generator')
print(f'did not enforce uniqueness at the grain level and produced the same combination')
print(f'multiple times with randomly varied measure values.')
print(f'\nEffect on analysis: minimal. Tableau aggregates using AVG across these rows,')
print(f'which smooths out the random variation and produces a reasonable central estimate.')

**Finding:** The closest approximation to a grain is `Job Title + Industry + Location + Experience Required (Years)`, which reduces duplicates from 97.9% to just 1.7%. The remaining 513 duplicate rows appear to be a synthetic generation artefact — identical grain combinations with randomly varied measure values. There is no additional dimension in the dataset that resolves this. The effect on analysis is minimal as aggregation (AVG) smooths out the variation.

## 4. Data Quality Finding: Job Status Field Inconsistency

The dataset contains a `Job Status` field labelled "Increasing" or "Decreasing". We check whether this label is consistent with the actual direction of change between `Job Openings (2024)` and `Projected Openings (2030)`.

In [ ]:
# Derive actual job status from the numeric openings fields
df['Actual Job Status'] = df.apply(
    lambda row: 'Increasing' if row['Projected Openings (2030)'] > row['Job Openings (2024)'] else 'Decreasing',
    axis=1
)

# Compare with the original Job Status field
contradictions = df[df['Job Status'] != df['Actual Job Status']]

print(f'Total rows: {len(df)}')
print(f'Contradictions: {len(contradictions)} ({len(contradictions)/len(df)*100:.1f}% of rows)')
print(f'Consistent rows: {len(df) - len(contradictions)} ({(len(df) - len(contradictions))/len(df)*100:.1f}% of rows)')

In [ ]:
# Show a sample of contradicting rows
print('Sample contradictions (Job Status label vs actual direction):')
contradictions[['Job Title', 'Industry', 'Location', 'Job Status', 'Job Openings (2024)', 
                'Projected Openings (2030)', 'Actual Job Status']].head(10)

In [ ]:
# Visualise the contradiction distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = pd.Series({'Consistent': len(df) - len(contradictions), 'Contradicting': len(contradictions)})
colors = ['#4CAF50', '#F44336']
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Job Status Field: Consistency Check', fontsize=13, pad=12)
ax.set_ylabel('Number of rows')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

**Finding:** Nearly 50% of rows (14,924 out of 30,000) contain a contradiction between the `Job Status` label and the actual direction of change implied by the openings figures. The `Job Status` field was therefore replaced in the Tableau analysis with a derived field based directly on the numeric comparison.

**Note on discrepancy with Tableau:** An earlier check in Tableau suggested only 25% contradictions. The difference is explained by aggregation — Tableau compared Job Status against *averaged* openings figures across multiple rows per job title, which smoothed out some contradictions. Python checks row by row, which is more granular and reveals the true extent of the inconsistency. The row-level Python figure (49.7%) is the more accurate one.

This appears to be a result of the synthetic data generation process assigning `Job Status` independently rather than deriving it from the openings data.

## 5. Deriving the Total Structural Change Metric

Net job change alone can be misleading. A near-zero net figure can mask enormous structural disruption if large numbers of jobs are simultaneously being created and destroyed. The **Total Structural Change** metric captures the total volume of workforce displacement regardless of direction.

In [ ]:
# Calculate net job change and structural change
df['Net Job Change'] = df['Projected Openings (2030)'] - df['Job Openings (2024)']
df['Structural Change'] = df['Net Job Change'].abs()

# Global totals
total_jobs_2024 = df['Job Openings (2024)'].sum()
net_change = df['Net Job Change'].sum()
structural_change = df['Structural Change'].sum()

print(f'Total Job Market 2024:     {total_jobs_2024:>15,.0f}')
print(f'Net Job Change:            {net_change:>+15,.0f}  ({net_change/total_jobs_2024*100:.2f}%)')
print(f'Total Structural Change:   {structural_change:>15,.0f}  ({structural_change/total_jobs_2024*100:.1f}%)')

In [ ]:
# Germany subset
germany = df[df['Location'] == 'Germany']

total_jobs_de = germany['Job Openings (2024)'].sum()
net_change_de = germany['Net Job Change'].sum()
structural_change_de = germany['Structural Change'].sum()

print('Germany:')
print(f'Total Job Market 2024:     {total_jobs_de:>15,.0f}')
print(f'Net Job Change:            {net_change_de:>+15,.0f}  ({net_change_de/total_jobs_de*100:.2f}%)')
print(f'Total Structural Change:   {structural_change_de:>15,.0f}  ({structural_change_de/total_jobs_de*100:.1f}%)')

In [ ]:
# Structural change vs net change by industry (Germany)
germany_industry = germany.groupby('Industry').agg(
    Net_Job_Change=('Net Job Change', 'sum'),
    Structural_Change=('Structural Change', 'sum')
).sort_values('Structural_Change', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Net job change by industry
sorted_net = germany_industry['Net_Job_Change'].sort_values()
colors_net = ['#4CAF50' if x > 0 else '#F44336' for x in sorted_net]
sorted_net.plot(
    kind='barh', ax=axes[0], color=colors_net, edgecolor='white'
)
axes[0].set_title('Net Job Change by Industry\n(Germany)', fontsize=12)
axes[0].set_xlabel('Net Job Change (number of jobs)')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Structural change by industry
germany_industry['Structural_Change'].sort_values().plot(
    kind='barh', ax=axes[1], color='#2196F3', edgecolor='white'
)
axes[1].set_title('Total Structural Change by Industry\n(Germany)', fontsize=12)
axes[1].set_xlabel('Job Positions in Flux (millions)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

**Finding:** Germany's labour market appears near-stable on a net basis (-0.21%), but 65% of all jobs are in structural flux. This means millions of workers will need to transition between roles — a challenge that net job change figures alone would completely obscure. The finance sector shows the largest net job losses and should be the primary focus of reskilling programmes.

---

## Summary

| Finding | Detail |
|---------|--------|
| Approximate data grain | Job Title + Industry + Location + Experience Required (Years) |
| Residual duplicates at grain level | 513 rows (1.7%) — synthetic generation artefact, minimal effect on AVG aggregations |
| Job Status contradictions | 14,924 rows (49.7%) at row level — field replaced with derived Actual Job Status. Note: Tableau showed ~25% due to aggregation smoothing. |
| Germany net job change | approx -0.21% of total market |
| Germany structural change | approx 65% of total market |
| Automation risk | approx 50% average across all industries |

For the full interactive analysis, see the **[Tableau Story](https://public.tableau.com/app/profile/robert.wilson8303/viz/ai_jobmarket/AITheFutureofWorkADataStory)**.

*Robert W. — March 2026*